# 01 Manual Exploration

This notebook is the first exploration step before the larger tagging and modeling pipeline.

At this point, I am not trying to build the final dashboard yet.  
I am trying to inspect early review patterns and decide what kinds of reviews are worth manually tagging.

The exploration focuses on questions like:

- Which negative reviews come from experienced players?
- Are there `Recommended` reviews that still mention bugs or complaints?
- Do funny / meme-like reviews appear in the data?
- Are there reviews that sound nostalgic, disappointed, or like the game “used to be better”?
- Which reviews should be exported as the first manual tagging candidate file?

This notebook is part of the **pilot tagging stage**, so it uses the older processed review file that has been moved into `data/archive/`.

## 1. Load the pilot processed review data

The original version of this notebook loaded:

```text
../data/processed/phasmophobia_reviews_processed.csv
```

After cleaning the repo structure, that file is now placed in:

```text
../data/archive/phasmophobia_reviews_processed.csv
```

This source file is kept because the notebook belongs to the earlier pilot exploration stage.  
Changing it to the expanded processed dataset would change the exported tagging candidates and may affect the later sequential process.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_ARCHIVE = Path("../data/archive")
DATA_INTERIM = Path("../data/interim")

# Load the pilot processed dataset used by this notebook
df = pd.read_csv(DATA_ARCHIVE / "phasmophobia_reviews_processed.csv")

# Quick preview
df.head(3)

,game_title,author_name,review_date,recommendation,playtime_raw,playtime_hours,review_length,helpful_text,review_text,cleaned_review,helpful_count,funny_count,word_count,char_count,contains_update_keyword,contains_bug_keyword,contains_emotional_keyword,is_veteran,playtime_segment
0,Phasmophobia,𝓜𝓸𝓷𝓼𝓽𝓮𝓻,Posted: 12 May,Recommended,5.3 hrs on record,5.3,58,No one has rated this review as helpful yet 0,Posted: 12 May\nEARLY ACCESS REVIEW\nYou can r...,You can roast the ghost,0,0,5,23,False,False,False,False,New
1,Phasmophobia,Oyster,Posted: 12 May,Recommended,72.7 hrs on record,72.7,119,No one has rated this review as helpful yet 0,Posted: 12 May\nEARLY ACCESS REVIEW\nits an al...,its an alright game but for me its too clumsy ...,0,0,19,84,False,True,False,False,Regular
2,Phasmophobia,Clown,Posted: 12 May,Not Recommended,77.1 hrs on record,77.1,124,No one has rated this review as helpful yet 0,Posted: 12 May\nEARLY ACCESS REVIEW\nRushed ou...,Rushed out a buggy and broken update so they c...,0,0,17,89,True,True,False,False,Regular


## 2. Experienced players with negative reviews

First, this step checks whether highly experienced players are writing longer `Not Recommended` reviews.

These reviews are useful because they may contain deeper complaints than short casual reviews.  
For example, veteran players may mention long-term gameplay changes, bugs, or disappointment after updates.

In [2]:
df[
    (df['is_veteran'] == True) &
    (df['recommendation'] == 'Not Recommended')
][[
    'playtime_hours',
    'word_count',
    'cleaned_review'
]].sort_values(
    by='word_count',
    ascending=False
).head(10)

,playtime_hours,word_count,cleaned_review
17,286.1,490,"TLDR; New gamers, I urge you not to buy this g..."
12,476.6,418,"I used to really love playing this game, but i..."
37,162.8,179,So were supposed to get character customizatio...
38,125.5,166,Phas had a special jankyness that brought a li...
16,470.5,149,TL;DR Don't get it yet character customization...
27,630.3,105,Boycott until the jank is fixed. They company ...
15,126.2,103,"The ""Player Character Update 0.17.0.0"" has sho..."
28,777.7,57,"This was a great game, an ""indie"" that made it..."
36,101.0,57,Multi-million dollars studio that acts aimless...
7,215.6,31,this game was great until the character update...


## 3. Recommended reviews that still mention bugs

Next, I look for a possible early mismatch pattern.

Some reviews may be marked as `Recommended`, but the text still talks about bugs or glitches.  
These reviews are interesting because the Steam label looks positive, while the actual text may be more mixed.

In [ ]:
# Recommended reviews that still mention bugs or glitches
mismatch_suspects = df[
    (df['recommendation'] == 'Recommended') & 
    (df['contains_bug_keyword'] == True)
][['playtime_hours', 'cleaned_review', 'helpful_count']]

mismatch_suspects.head(10)

,playtime_hours,cleaned_review,helpful_count
1,72.7,its an alright game but for me its too clumsy ...,0
8,25.6,I'm really enjoying this new update. The diner...,0
10,37.1,amazing game. 10/10. they do need to fix the b...,0


## 4. Funny or joke-like reviews

Steam reviews are not always written as serious feedback.

Here, I sort reviews by `funny_count` to see which reviews were treated as jokes by the community.  
These reviews may become difficult to classify later because the sentiment is not always clear from the text alone.

In [5]:
# Reviews with the highest Funny vote count
funny_reviews = df.sort_values(by='funny_count', ascending=False)[
    ['recommendation', 'funny_count', 'cleaned_review']
]

funny_reviews.head(15)

,recommendation,funny_count,cleaned_review
29,Not Recommended,1,"Weird animations, broken game, ♥♥♥♥♥♥ models r..."
39,Not Recommended,1,Super disappointed at the new update. I've bee...
37,Not Recommended,1,So were supposed to get character customizatio...
33,Not Recommended,1,6 years and tons of money yet they can't get t...
0,Recommended,0,You can roast the ghost
1,Recommended,0,its an alright game but for me its too clumsy ...
2,Not Recommended,0,Rushed out a buggy and broken update so they c...
3,Recommended,0,I'm a lot less than just cleaning supplies and...
7,Not Recommended,0,this game was great until the character update...
6,Recommended,0,Product refunded I liked this game but I didn'...


## 5. Reviews with mourning / nostalgia signals

Some reviews may express that the game used to be better.

This is not necessarily a simple positive or negative statement.  
It can show disappointment, nostalgia, or frustration with how the game changed over time.

Here, I use simple keywords to surface possible examples for later manual inspection.

In [6]:
# Define exploratory keywords for nostalgia or disappointment
mourning_keywords = ['used to', 'miss', 'ruined', 'great game', 'was fun', 'they killed']

# Create a temporary flag for reviews that contain nostalgia/disappointment keywords
df['is_mourning'] = df['cleaned_review'].apply(
    lambda x: any(kw in str(x).lower() for kw in mourning_keywords)
)

mourning_reviews = df[df['is_mourning'] == True][
    ['recommendation', 'playtime_hours', 'cleaned_review']
]

mourning_reviews.head(10)

,recommendation,playtime_hours,cleaned_review
5,Not Recommended,23.5,The latest update has ruined the experience fo...
8,Recommended,25.6,I'm really enjoying this new update. The diner...
12,Not Recommended,476.6,"I used to really love playing this game, but i..."
19,Recommended,33.9,"great game, some players are so whiny about up..."
23,Recommended,294.2,Great game and great devs.
27,Not Recommended,630.3,Boycott until the jank is fixed. They company ...
28,Not Recommended,777.7,"This was a great game, an ""indie"" that made it..."
36,Not Recommended,101.0,Multi-million dollars studio that acts aimless...
39,Not Recommended,32.7,Super disappointed at the new update. I've bee...


## 6. Export the first manual tagging candidate file

After checking the early patterns, I export a mixed set of reviews for manual tagging.

The sample contains:

1. `Recommended` reviews that mention bugs  
2. other `Recommended` reviews  
3. `Not Recommended` reviews  

This output is saved to `data/interim/` because it is a tagging preparation file, not a final processed dataset.

In [7]:
# Add empty columns for manual tagging
df['actual_sentiment'] = ''
df['review_category'] = ''

# Build a mixed candidate sample: possible mismatch, positive-looking reviews, and negative reviews
# If a group has fewer than 30 rows, head(30) will return the available rows.

sample_mismatch = df[(df['recommendation'] == 'Recommended') & (df['contains_bug_keyword'] == True)].head(30)
sample_positive = df[(df['recommendation'] == 'Recommended') & (df['contains_bug_keyword'] == False)].head(30)
sample_negative = df[df['recommendation'] == 'Not Recommended'].head(30)

# Combine all candidate groups into one tagging dataset
df_to_tag = pd.concat([sample_mismatch, sample_positive, sample_negative])

# Export the candidate set for manual tagging
df_to_tag.to_csv(DATA_INTERIM / "data_for_manual_tagging.csv", index=False)
print(f"Saved manual tagging candidate file to: {DATA_INTERIM / 'data_for_manual_tagging.csv'}")


Saved manual tagging candidate file to: ..\data\interim\data_for_manual_tagging.csv


## 7. Exploration note

This notebook produces the first candidate tagging file:

```text
data/interim/data_for_manual_tagging.csv
```

The file is part of the early manual tagging process.  
Later notebooks expand the dataset, combine tagging batches, prepare model-ready data, and generate dashboard-ready CSV files.

Because this notebook belongs to the pilot stage, I intentionally keep the original logic and only update the file routes to match the cleaned project structure.
